<a href="https://colab.research.google.com/github/deivid-cp/tfg/blob/main/setting_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import zipfile
import os

# Paths of the directory
PROJECT_ROOT_DIR = os.getcwd()
PROJECT_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, "proyecto")
IMAGES_PATH = os.path.join(PROJECT_ROOT_DIR, "images")
DATA_PATH = os.path.join(PROJECT_ROOT_DIR, "datasets")

In [6]:
# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Where to save the figures
os.makedirs(IMAGES_PATH, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(IMAGES_PATH, fig_id + "." + fig_extension)
    print("Saving figure", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

In [7]:
# Where to unzip data
os.makedirs(DATA_PATH, exist_ok=True)

def unzip_data(zipfile_name, directory_to_extract_to = DATA_PATH):

    path_to_zip_file = os.path.join(PROJECT_ROOT_DIR, zipfile_name)

    with zipfile.ZipFile(path_to_zip_file, 'r') as zip_ref:
        zip_ref.extractall(directory_to_extract_to)

unzip_data(os.path.join(PROJECT_ROOT_DIR, "pan21-author-profiling-test-2021-04-12.zip"))
unzip_data(os.path.join(PROJECT_ROOT_DIR, "pan21-author-profiling-training-2021-03-14.zip"))

In [8]:
# Data processing
# Extracting data from .xml files to Pandas DataFrame and saving them as .csv

import pandas as pd
import xml.etree.ElementTree as ET

# A list of two components with dictionaries in each of them with which we will
# create the dataframes, in the first component the training set and in the
# second one the test set
L = [{'user' : [], 'raw_text' : [], 'language' : [], 'class' : []},
 {'user' : [], 'raw_text' : [], 'language' : [], 'class' : []}]

# A list with the directories we will iterate, the last component is the current
# directory and the previous ones the ones we have already iterated to reach the
# current one
paths_list = [DATA_PATH]

for dir1 in os.listdir(paths_list[0]):    # os.listdir(path) returns a list of the
                                          # names of the files inside the given path
    if dir1 == 'pan21-author-profiling-training-2021-03-14':
        set_type = 0    # The training set will be in the first component
    elif dir1 == 'pan21-author-profiling-test-2021-04-12':
        set_type = 1    # The test set will be in the second component
    else:    # In order to skip the file '.ipynb_checkpoints' created by Jupyter
        continue
    paths_list.append(os.path.join(paths_list[-1], dir1))
    for dir2 in os.listdir(paths_list[-1]):
        if dir2 == "en":
            lng = 'english'
        else:
            lng = 'spanish'
        paths_list.append(os.path.join(paths_list[-1], dir2))
        for user in os.listdir(paths_list[-1]):
            if user == '.ipynb_checkpoints' or user == 'truth.txt':
                continue
            tree = ET.parse(os.path.join(paths_list[-1], user))
            root = tree.getroot()

            tweets = []

            # The root attributes are a dictionary and the value of the key 'class'
            # is a string with a '0' (no hate speech) or a '1' (hate speech)
            cls = int(root.attrib['class'])

            for documents in root:
                for document in documents:
                    tweets.append(document.text)
            L[set_type]['user'].append(user)
            L[set_type]['raw_text'].append(tweets)
            L[set_type]['language'].append(lng)
            L[set_type]['class'].append(cls)
        del paths_list[-1]
    del paths_list[-1]

# We create the whole training and test datasets and save them in a .csv
training_set = pd.DataFrame(L[0])
test_set = pd.DataFrame(L[1])
training_set.to_csv(os.path.join(DATA_PATH, "training_set.csv"), index=False)
test_set.to_csv(os.path.join(DATA_PATH, "test_set.csv"), index=False)

# We create the english training and test datasets and save them in a .csv
training_set_english = training_set.loc[training_set['language'] == 'english']
test_set_english = test_set.loc[test_set['language'] == 'english']
training_set_english.to_csv(os.path.join(DATA_PATH, "training_set_english.csv"), index=False)
test_set.to_csv(os.path.join(DATA_PATH, "test_set_english.csv"), index=False)

# We create the spanish training and test datasets and save them in a .csv
training_set_spanish = training_set.loc[training_set['language'] == 'spanish']
test_set_spanish = test_set.loc[test_set['language'] == 'spanish']
training_set_spanish.to_csv(os.path.join(DATA_PATH, "training_set_spanish.csv"), index=False)
test_set_spanish.to_csv(os.path.join(DATA_PATH, "test_set_spanish.csv"), index=False)

In [9]:
training_set.head()

,user,raw_text,language,class
0,9cff4936f8479d53fcbb63f2524c5ad8.xml,[It’s pissing me off how desperate Paige is wt...,english,0
1,e30b87c8fa34a1ac077d7b286000bc06.xml,"[Ima be 20 hrs from Houston now ugh, Fuga pa V...",english,1
2,f91fa8ecdd2440eb163516769573f24a.xml,[me and kayleigh just waking up lol we finna b...,english,1
3,44e25d1ec3f786a696e0eb5b33b8325b.xml,"[WHO THE OG'S? #URL#, ATE THE WRONG PUSSY GOTT...",english,1
4,7c18d72ba1eb9c787f166df788e1b521.xml,[if somone told me that you could teach a 15 m...,english,0


In [10]:
training_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   user      400 non-null    object
 1   raw_text  400 non-null    object
 2   language  400 non-null    object
 3   class     400 non-null    int64 
dtypes: int64(1), object(3)
memory usage: 12.6+ KB


In [11]:
test_set.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   user      200 non-null    object
 1   raw_text  200 non-null    object
 2   language  200 non-null    object
 3   class     200 non-null    int64 
dtypes: int64(1), object(3)
memory usage: 6.4+ KB


In [12]:
training_set_english.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200 entries, 0 to 199
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   user      200 non-null    object
 1   raw_text  200 non-null    object
 2   language  200 non-null    object
 3   class     200 non-null    int64 
dtypes: int64(1), object(3)
memory usage: 7.8+ KB


In [13]:
training_set_spanish.info()

<class 'pandas.core.frame.DataFrame'>
Index: 200 entries, 200 to 399
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   user      200 non-null    object
 1   raw_text  200 non-null    object
 2   language  200 non-null    object
 3   class     200 non-null    int64 
dtypes: int64(1), object(3)
memory usage: 7.8+ KB


In [14]:
test_set_english.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   user      100 non-null    object
 1   raw_text  100 non-null    object
 2   language  100 non-null    object
 3   class     100 non-null    int64 
dtypes: int64(1), object(3)
memory usage: 3.9+ KB


In [15]:
test_set_spanish.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 100 to 199
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   user      100 non-null    object
 1   raw_text  100 non-null    object
 2   language  100 non-null    object
 3   class     100 non-null    int64 
dtypes: int64(1), object(3)
memory usage: 3.9+ KB


In [16]:
training_set['raw_text']

,raw_text
0,[It’s pissing me off how desperate Paige is wt...
1,"[Ima be 20 hrs from Houston now ugh, Fuga pa V..."
2,[me and kayleigh just waking up lol we finna b...
3,"[WHO THE OG'S? #URL#, ATE THE WRONG PUSSY GOTT..."
4,[if somone told me that you could teach a 15 m...
...,...
395,"[Paolo Maldini, aishh que hombre 😍😍 La eleganc..."
396,[Honor a los Leales. Y a nuestros Caídos: un f...
397,[Si van a hacer públicos los citados de las ca...
398,"[muero de amor con milo 🥰, vieron cuando les d..."


In [17]:
training_set['raw_text'][0]

['It’s pissing me off how desperate Paige is wtfffffff #HASHTAG#',
 'Being wholesome 💫 #URL#',
 'Lmfaooooooooooooooooo I’m imagining wtf she doing #URL#',
 '#USER# I’m waiting on 2 more final exam grades lmaoooo lettuce pray!',
 'RT #USER#: When I die please don’t text me “tell me it’s not true😭” cuz I will not reply.',
 'RT #USER#: When they ask me how many LLCs ima register w/ my stimmy #URL#',
 'Last term grades rolling in. I really got a 100 on a final 😂😂😂 I deserve....something....',
 'Could y’all fucking pleaseeeeee 😭😭😭😭😭😭😭😭😭😭😭😭😭 #URL#',
 'RT #USER#: I ain’t even complaining about gas nomo.. This $20 going ina tank whether it’s 1.99 or 2.50...😂😂',
 'RT #USER#: Same thing w having kids . Everybody shouldnt be allowed to have one. U gotta be able to qualify for that',
 'RT #USER#: &amp; give some of y’all children their stimmy money so they can stop breaking in ppl cars plz. stop being greedy!',
 'RT #USER#: Phase 3 in New Orleans? Boy they bout to have to tear gas niggas to make t